# Stpm Exploration
Acquire data, inspect coverage and distributions, build PostGIS, and explore a tract-quarter cube.

Both notebooks use `crime.downloader`, `importer`, `generator`, `warehouse`,
`stpm`, `validation`, `visualizer` and `exporter`. SSP and SINESP remain distinct
sources with explicit spatial and temporal capabilities.

[Base dos Dados SINESP catalogue](https://basedosdados.org/dataset/7f6f7de2-f4e9-467c-a8c4-a66a6559803d)
· [Colab form fields](https://colab.research.google.com/notebooks/forms.ipynb)
· [Package architecture](../docs/PACKAGE_ARCHITECTURE.md)


## 1. Workspace and package
Run cells in order. Colab form fields are editable parameters; in Jupyter, edit
these same Python assignments. Use Python 3.12 or newer. Upload the built wheel and set PACKAGE_SOURCE to its
path, or use an already installed package. No Git clone is required.

Colab storage and a temporary PostGIS database disappear when the runtime ends.
Choose a workspace on mounted Drive, or download the final results archive.
Installing the package can require restarting the runtime if Colab has already
imported incompatible versions; rerun setup after restarting.


In [ ]:
# @title Workspace and installation
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = "" # @param {type:"string"}
STUDY_DATA_DIRECTORY = "" # @param {type:"string"}
PACKAGE_SOURCE = "" # @param {type:"string"}
INSTALL_PACKAGE = True # @param {type:"boolean"}

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
ROOT = Path(PROJECT_ROOT).expanduser() if PROJECT_ROOT else Path.cwd() / "crime-workspace"
ROOT.mkdir(parents=True, exist_ok=True)
os.environ["CRIME_PROJECT_ROOT"] = str(ROOT.resolve())
if STUDY_DATA_DIRECTORY:
    os.environ["CRIME_DATA_DIR"] = str(Path(STUDY_DATA_DIRECTORY).expanduser())
if INSTALL_PACKAGE and PACKAGE_SOURCE:
    subprocess.run([sys.executable, "-m", "pip", "install", PACKAGE_SOURCE + "[etl,db,verification]"], check=True)
if importlib.util.find_spec("crime") is None:
    raise RuntimeError("Upload the package wheel and set PACKAGE_SOURCE to its path, then rerun setup")
from IPython.display import Markdown, display

import crime

print("Package:", crime.__version__, "| workspace:", ROOT)


## 2. Sources and study parameters
**SSP** provides the pinned Campinas 2022–2025 report-level spatial study.
**Base dos Dados / SINESP** provides municipal monthly indicators: temporal
analysis is available, but tract maps, LISA, Gi* and EHSA are unavailable.

Demographic source selection applies to the **municipal population reference**.
The spatial study still requires IBGE tract polygons and attributes and CNEFE
addresses. A municipal population value cannot replace tract populations.
Reference dates are separate from the fixed spatial-study window.


In [ ]:
# @title Data sources and Base dos Dados parameters
NOTEBOOK_NAME = "stpm_exploration"
REPORT_SOURCE = "ssp" # @param ["ssp", "basedosdados_sinesp"]
DEMOGRAPHIC_REFERENCE = "none" # @param ["none", "basedosdados_population"]
ENABLE_BASE_DOS_DADOS = False # @param {type:"boolean"}
GOOGLE_PROJECT_ID = "campinas-crime-stpm" # @param {type:"string"}
MUNICIPALITY = "3509502" # @param {type:"string"}
REFERENCE_START_YEAR = 2015 # @param {type:"integer"}
REFERENCE_END_YEAR = 2025 # @param {type:"integer"}
MAXIMUM_BYTES_BILLED = 1000000000 # @param {type:"integer"}
REPORTING_INSTITUTION = "Estadual" # @param {type:"string"}
REFERENCE_MEASURE = "quantidade_ocorrencias" # @param ["quantidade_ocorrencias", "quantidade_vitimas"]
DOWNLOAD_SSP_INPUTS = True # @param {type:"boolean"}
INPUT_RELEASE_URL = "" # @param {type:"string"}
USE_STAGED_SSP = True # @param {type:"boolean"}

USE_BD = ENABLE_BASE_DOS_DADOS or REPORT_SOURCE == "basedosdados_sinesp" or DEMOGRAPHIC_REFERENCE != "none"
if REPORT_SOURCE == "ssp" and MUNICIPALITY != "3509502":
    raise ValueError("The pinned report-level treatments currently support Campinas only")
from datetime import datetime, timezone

RUN_DIR = ROOT / "outputs/guided" / (NOTEBOOK_NAME + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Results:", RUN_DIR)


## 3. Authenticate and preview external queries
Run this cell to open Google's authentication prompt in Colab. Locally,
configure Application Default Credentials with `gcloud auth application-default login`.
Tokens and database passwords are never printed or stored in notebook fields.
The project ID selects the project that runs the query, not the data host.

A dry run estimates the data scanned; the following extraction cell is controlled
separately. The byte cap applies to each executed query. Catalogue coverage is
not proof of complete Campinas observations.


In [ ]:
# @title Authenticate and estimate the queries
AUTHENTICATE_GOOGLE = True # @param {type:"boolean"}
from crime.downloader.basedosdados import (
    ReferenceQuery,
    authenticate_google,
    inspect_reference_schema,
    read_population,
    read_sinesp,
)

request = None
if USE_BD:
    request = ReferenceQuery(GOOGLE_PROJECT_ID, MUNICIPALITY, REFERENCE_START_YEAR,
                             REFERENCE_END_YEAR, MAXIMUM_BYTES_BILLED)
    if AUTHENTICATE_GOOGLE:
        authenticate_google(GOOGLE_PROJECT_ID)
    display(inspect_reference_schema(request).data)
    display(read_sinesp(request, dry_run=True))
    if DEMOGRAPHIC_REFERENCE != "none":
        display(read_population(request, dry_run=True))
else:
    print("External queries disabled. Enable Base dos Dados above to authenticate and compare sources.")


In [ ]:
# @title Download external reference snapshots
EXECUTE_REFERENCE_QUERIES = True # @param {type:"boolean"}
sinesp_snapshot = population_snapshot = reference_series = None
if USE_BD and EXECUTE_REFERENCE_QUERIES:
    from crime.importer.basedosdados import municipal_series
    sinesp_snapshot = read_sinesp(request)
    sinesp_snapshot.save(RUN_DIR / "references", "sinesp")
    display(sinesp_snapshot.data.head(20))
    display(sinesp_snapshot.data.groupby(["abrangencia", "tipo_ocorrencia"]).size().rename("rows").to_frame())
    reference_series = municipal_series(
        sinesp_snapshot.data, institution=REPORTING_INSTITUTION, measure=REFERENCE_MEASURE,
        municipality=MUNICIPALITY, start=f"{REFERENCE_START_YEAR}-01-01", end=f"{REFERENCE_END_YEAR}-12-31")
    display(reference_series.coverage.apply(lambda col: col.value_counts()).fillna(0).astype(int))
    display(reference_series.values.describe().T)
    if DEMOGRAPHIC_REFERENCE != "none":
        population_snapshot = read_population(request)
        population_snapshot.save(RUN_DIR / "references", "population_2022")
        display(population_snapshot.data)
elif REPORT_SOURCE == "basedosdados_sinesp":
    raise RuntimeError("Execute reference queries to use Base dos Dados as the analysis source")


## 4. Acquire and treat the spatial-study inputs
Existing staged data can be reused. Otherwise the package applies its pinned
acquisition, cleaning and tract-assignment rules. Missing inputs can be downloaded
from an explicit release URL exposing the paths in the packaged `campinas-2022-2025.sha256` manifest.
The package verifies all checksums and never substitutes aggregate counts for
individual reports. If inputs are private, mount/upload them into STUDY_DATA_DIRECTORY before importing the package.


In [ ]:
# @title Prepare SSP reports, IBGE tracts and CNEFE addresses
import json

import pandas as pd

from crime.config import INPUTS_DIR, STAGING_DIR
from crime.generator import generate_dataset, load_dataset

dataset = None
if REPORT_SOURCE == "ssp":
    if USE_STAGED_SSP and (STAGING_DIR / "units/bulletins.parquet").is_file():
        dataset = load_dataset()
    else:
        from crime.downloader.study import download_study_inputs
        from crime.importer.acquisition import expected_checksums
        checksums = expected_checksums()
        missing = [name for name in checksums if not (INPUTS_DIR / name).is_file()]
        if missing:
            if not DOWNLOAD_SSP_INPUTS or not INPUT_RELEASE_URL:
                raise RuntimeError("Missing study inputs. Set INPUT_RELEASE_URL or upload: " + ", ".join(missing))
            download_study_inputs(INPUT_RELEASE_URL, INPUTS_DIR, checksums)
        dataset = generate_dataset()
    import json
    from dataclasses import asdict
    display({"reports": len(dataset.bulletins), "tracts": len(dataset.tracts), "addresses": len(dataset.poi)})
    if dataset.reports:
        import pandas as pd
        display(pd.DataFrame([asdict(r) for r in dataset.reports]))
    (RUN_DIR / "data_sources.json").write_text(json.dumps([asdict(s) for s in dataset.sources], indent=2, default=str))
else:
    print("Municipal aggregate source selected: tract-level spatial analyses are unavailable.")


## 5. Warehouse
`auto` starts an isolated temporary PostGIS server in Colab and uses your configured
external database locally. `none` keeps exploration in memory. Temporary setup
installs PostgreSQL/PostGIS in the Colab runtime; it does not expose a public port.

Existing study tables are preserved unless **REPLACE_EXISTING_WAREHOUSE** is
selected. A reused warehouse is an existing release; this notebook's calculations
use the selected in-memory dataset, whose provenance is saved in this run.
External reference snapshots are stored in a separate schema, never bulletin facts.


In [ ]:
# @title Build or connect to PostGIS
WAREHOUSE_MODE = "auto" # @param ["auto", "temporary_colab", "external", "none"]
REPLACE_EXISTING_WAREHOUSE = False # @param {type:"boolean"}
PG_HOST = "localhost" # @param {type:"string"}
PG_PORT = 5433 # @param {type:"integer"}
PG_DATABASE = "crime_db" # @param {type:"string"}
PG_USER = "admin" # @param {type:"string"}

warehouse_config = None
mode = ("temporary_colab" if IN_COLAB else "external") if WAREHOUSE_MODE == "auto" else WAREHOUSE_MODE
if mode != "none":
    from sqlalchemy import text

    from crime.config import WarehouseConfig, engine
    from crime.warehouse.notebook import load_notebook_warehouse, start_colab_postgis
    from crime.warehouse.references import store_reference
    if mode == "temporary_colab":
        warehouse_config = start_colab_postgis()
    else:
        from getpass import getpass
        # Password input is hidden and does not become a saved form parameter.
        password = os.environ.get("PGPASSWORD") or os.environ.get("POSTGRES_PASSWORD") or getpass("PostGIS password: ")
        warehouse_config = WarehouseConfig(host=PG_HOST, port=PG_PORT, database=PG_DATABASE, user=PG_USER, password=password)
        del password
    db_engine = engine(warehouse_config)
    try:
        with db_engine.connect() as connection:
            existing = connection.execute(text("SELECT to_regclass('public.build') IS NOT NULL")).scalar_one()
        if dataset is not None:
            if not existing or REPLACE_EXISTING_WAREHOUSE:
                loaded = load_notebook_warehouse(dataset, warehouse_config, replace_existing=REPLACE_EXISTING_WAREHOUSE)
                display(loaded.extra["funnel"])
            else:
                print("Existing warehouse preserved; notebook analyses use the selected dataset.")
            from crime.warehouse import Warehouse
            with Warehouse(warehouse_config) as wh:
                display(wh.build())
        for snapshot in (sinesp_snapshot, population_snapshot):
            if snapshot is not None:
                print("Stored reference snapshot:", store_reference(db_engine, snapshot))
    finally:
        db_engine.dispose()
else:
    print("In-memory mode: no database is created or modified.")


## 6. Reconcile external references
Population agreement concerns the municipal total only. Crime reconciliation
requires a reviewed crosswalk from a SINESP indicator to local **primary-offence
report** slugs, plus confirmation that date and counting definitions match.
Victim totals cannot be compared with report counts. All local reports are used,
including reports without coordinates. Missing or structural-zero reference
months remain unavailable; percentages relative to a zero reference are undefined.

Enter JSON such as `{"source_indicator": {"crime_slugs": ["local_slug"],
"definition_note": "Document why counting and category definitions match"}}`.
Inspect the actual source indicators and local taxonomy before filling it in.
No crosswalk is inferred from similar names.


In [ ]:
# @title Population and crime reconciliation
CROSSWALK_JSON = "{}" # @param {type:"string"}
DEFINITIONS_CONFIRMED = False # @param {type:"boolean"}
from crime.validation.reconciliation import IndicatorMapping, compare_population, reconcile_reports

comparison = None
if dataset is not None and population_snapshot is not None:
    population_check = compare_population(dataset.tracts, population_snapshot.data)
    display(population_check)
    (RUN_DIR / "population_reconciliation.json").write_text(json.dumps(population_check, indent=2))
if dataset is not None and reference_series is not None:
    from crime.taxonomy import SLUG_TO_GROUP
    display(pd.DataFrame(sorted(SLUG_TO_GROUP.items()), columns=["local_slug", "offence_group"]))
    crosswalk = json.loads(CROSSWALK_JSON)
    if crosswalk:
        from dataclasses import replace
        # Limit comparison to the period actually covered by the local report release.
        values = reference_series.values.loc["2022-01-01":"2025-12-01"]
        reference_overlap = replace(reference_series, values=values, coverage=reference_series.coverage.reindex(values.index))
        mappings = [IndicatorMapping(indicator, tuple(item["crime_slugs"]), item["definition_note"])
                    for indicator, item in crosswalk.items()]
        comparison = reconcile_reports(dataset.bulletins, reference_overlap, mappings,
                                       definitions_confirmed=DEFINITIONS_CONFIRMED)
        comparison.to_csv(RUN_DIR / "crime_reconciliation.csv", index=False)
        display(comparison)
        from crime.visualizer.exploration import plot_reconciliation
        display(plot_reconciliation(comparison))
    else:
        print("Crime counts have not been reconciled: provide and review a category crosswalk above.")
else:
    print("To compare sources, select SSP and enable Base dos Dados in section 2.")


## 7. Exploratory data analysis
These are descriptive results. A high count is not itself a statistically
significant hotspot. Coverage tables keep the unmapped reports in the denominator.


In [ ]:
# @title Show descriptive tables and figures
import matplotlib.pyplot as plt
import pandas as pd

from crime.generator.workflow import analyze_reference, describe_dataset
from crime.visualizer.exploration import plot_exploration, plot_reference

if dataset is not None:
    descriptive_tables = describe_dataset(dataset)
    for name, table in descriptive_tables.items():
        display(Markdown("**" + name.replace("_", " ") + "**"))
        display(table.head(30))
        table.to_csv(RUN_DIR / (name + ".csv"), index=name == "tract_summary")
    for name, fig in plot_exploration(dataset).items():
        fig.savefig(RUN_DIR / (name + ".png"), dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
if reference_series is not None:
    display(reference_series.values)
    fig = plot_reference(reference_series)
    fig.savefig(RUN_DIR / "reference_overview.png", dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    reference_results = analyze_reference(reference_series, RUN_DIR / "reference_temporal")
    for name, table in reference_results.items():
        display(Markdown("**Reference " + name + "**"))
        display(table)


## Interactive space–time cube
Select a mapped offence group. The map and chart share a selected quarter;
click a tract to inspect its count history and significance before and after BH.
White means inference unavailable, not nonsignificant. Colours use fixed scales
across quarters. The exported HTML needs internet access for Leaflet and Plotly.
This viewer does not send data to a Python callback server.


In [ ]:
# @title Explore a tract-quarter cube
CUBE_GROUP = "all" # @param ["all", "property", "bodily_harm", "traffic", "drug", "violent"]
CALCULATE_SIGNIFICANCE_HISTORY = True # @param {type:"boolean"}
from crime.exporter.interactive import export_cube_html
from crime.generator import generate_cube
from crime.query import CrimeQuery
from crime.visualizer.cube import cube_explorer

if dataset is not None:
    cube = generate_cube(dataset, CrimeQuery(crime_group=CUBE_GROUP, time_unit="quarter"))
    cube_inference = None
    if CALCULATE_SIGNIFICANCE_HISTORY:
        from crime.geoprocessor.weights import build_w
        from crime.stpm.ehsa import analyze_cube
        units = dataset.tracts.sort_values("census_tract_id").rename(columns={"census_tract_id": "unit_id"})
        weights = build_w(units, "queen")
        cube_inference = analyze_cube(pd.DataFrame(cube.counts, index=cube.unit_ids, columns=cube.periods), weights)
    viewer = cube_explorer(cube, dataset.tracts,
                           bins=cube_inference.bins if cube_inference is not None else None,
                           labels=cube_inference.labels if cube_inference is not None else None)
    display(viewer)
    html_path = export_cube_html(viewer, RUN_DIR / ("cube_" + CUBE_GROUP + ".html"))
    print("Interactive export:", html_path)
else:
    print("The municipal reference has no tract coordinates; a tract-quarter cube cannot be constructed.")


## Save this run
The archive contains results, reference snapshots, query provenance, and any
interactive viewer or publication assets generated above. It does not include
Google tokens or database credentials. For a full workflow it may include the
persisted treated study dataset used for publication; choose where you store it.


In [ ]:
# @title Archive results
CREATE_ARCHIVE = True # @param {type:"boolean"}
DOWNLOAD_ARCHIVE = False # @param {type:"boolean"}
if CREATE_ARCHIVE:
    import shutil
    archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
    print("Saved:", archive)
    if DOWNLOAD_ARCHIVE and IN_COLAB:
        from google.colab import files
        files.download(archive)
